In [9]:
import pandas as pd

new_df = pd.read_csv('preprocessed_dataset.csv')
X = new_df.drop(columns=['Target_encoded'])
y = new_df['Target_encoded']

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Convert the dataframe to tensors
X_tensor = torch.tensor(X.values, dtype=torch.float32)

# Create a dataset and dataloader
batch_size = 64
dataset = TensorDataset(X_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [11]:
# Define VAE model
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super(VAE, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2_mean = nn.Linear(128, latent_dim)
        self.fc2_logvar = nn.Linear(128, latent_dim)
        self.fc3 = nn.Linear(latent_dim, 128)
        self.fc4 = nn.Linear(128, input_dim)
        
    def encode(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2_mean(h), self.fc2_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        h = torch.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h))
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

input_dim = X.shape[1]
vae = VAE(input_dim)

In [12]:
# Define the loss function
def loss_function(recon_x, x, mu, logvar):
    BCE = nn.functional.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

# Optimizer
optimizer = optim.Adam(vae.parameters(), lr=0.001)

In [13]:
# Train the VAE
epochs = 50
vae.train()
for epoch in range(epochs):
    train_loss = 0
    for batch in dataloader:
        data = batch[0]
        optimizer.zero_grad()
        recon_batch, mu, logvar = vae(data)
        loss = loss_function(recon_batch, data, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {train_loss/len(dataloader.dataset):.4f}')

Epoch 1, Loss: 11644.4286
Epoch 2, Loss: 11558.2196
Epoch 3, Loss: 11550.9917
Epoch 4, Loss: 11548.5375
Epoch 5, Loss: 11547.1242
Epoch 6, Loss: 11546.4793
Epoch 7, Loss: 11546.0687
Epoch 8, Loss: 11545.3653
Epoch 9, Loss: 11545.1425
Epoch 10, Loss: 11544.8676
Epoch 11, Loss: 11544.4103
Epoch 12, Loss: 11544.4093
Epoch 13, Loss: 11544.0872
Epoch 14, Loss: 11543.4994
Epoch 15, Loss: 11543.2404
Epoch 16, Loss: 11542.9663
Epoch 17, Loss: 11542.7325
Epoch 18, Loss: 11542.4343
Epoch 19, Loss: 11542.1544
Epoch 20, Loss: 11541.9759
Epoch 21, Loss: 11541.8453
Epoch 22, Loss: 11541.7281
Epoch 23, Loss: 11541.4712
Epoch 24, Loss: 11541.3268
Epoch 25, Loss: 11541.2791
Epoch 26, Loss: 11541.2071
Epoch 27, Loss: 11541.1274
Epoch 28, Loss: 11541.1621
Epoch 29, Loss: 11541.1811
Epoch 30, Loss: 11541.1786
Epoch 31, Loss: 11541.1712
Epoch 32, Loss: 11541.1728
Epoch 33, Loss: 11541.1594
Epoch 34, Loss: 11541.0410
Epoch 35, Loss: 11541.0165
Epoch 36, Loss: 11541.0174
Epoch 37, Loss: 11541.0362
Epoch 38, 

In [14]:
# Generate new synthetic samples
vae.eval()
num_samples_to_generate = 5000

with torch.no_grad():
    z = torch.randn(num_samples_to_generate, 32)  # 32 = latent_dim
    generated_samples = vae.decode(z).numpy()

# Create a dataframe for synthetic data
synthetic_df = pd.DataFrame(generated_samples, columns=X.columns)

# Clip values to [0,1] if needed (since we used sigmoid at the output)
synthetic_df = synthetic_df.clip(0, 1)

synthetic_df.head()

,Sequence_Length,pI,Basic_residues,Acidic_residues,Net_charge,Polar_residues,Hydrophobic_residues,Hydrophobicity,Boman_Index,Aliphatic_Index,...,yid,yk,yl,yq,yr,yt,ytk,yts,yv,yvl
0,1.041094e-07,0.508223,0.000007,6.149177e-08,0.394430,8.322746e-07,1.197623e-07,0.533334,0.619355,0.000095,...,0.000220,0.000302,0.000305,0.000331,0.000221,0.000233,0.000066,0.000104,0.000213,0.000437
1,4.022006e-01,0.000717,0.047361,4.351307e-01,0.001823,4.068205e-01,2.301829e-01,0.019683,0.031061,0.026751,...,0.002079,0.003848,0.004101,0.002645,0.002555,0.012051,0.002816,0.002712,0.004076,0.002596
2,2.894719e-05,0.522900,0.000288,2.025034e-05,0.420912,1.757734e-04,7.599363e-05,0.531317,0.593599,0.002390,...,0.002827,0.003078,0.002205,0.002191,0.002260,0.003165,0.001550,0.002077,0.002161,0.002909
3,5.578954e-01,0.000333,0.064234,7.270737e-01,0.000863,5.351668e-01,3.717232e-01,0.024446,0.044490,0.064545,...,0.003141,0.003182,0.002524,0.003386,0.002708,0.012084,0.002786,0.002585,0.003019,0.003139
4,7.027395e-06,0.261060,0.000084,6.286371e-06,0.185298,4.114756e-05,1.301507e-05,0.318406,0.470160,0.000640,...,0.000340,0.000805,0.001233,0.000628,0.000510,0.000811,0.000269,0.000186,0.001162,0.000834


In [15]:
import numpy as np

synthetic_labels = np.random.choice(y.unique(), size=len(synthetic_df))
synthetic_df['Target_encoded'] = synthetic_labels

# Combine original and synthetic data
augmented_df = pd.concat([new_df, synthetic_df], ignore_index=True)

print(f"Original dataset size: {new_df.shape}")
print(f"Augmented dataset size: {augmented_df.shape}")

Original dataset size: (2342, 514)
Augmented dataset size: (7342, 514)


In [16]:
# Save augmented dataset
augmented_df.to_csv('augmented_dataset.csv', index=False)

In [3]:
import pickle
import pandas as pd

data = pd.read_csv('preprocessed_dataset.csv')
with open('features.pkl', 'rb') as f:
    selected_features = pickle.load(f)
X = data.iloc[:, selected_features]

In [4]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(X)

synthesizer = CTGANSynthesizer(
    metadata=metadata,
    epochs=50,              
    batch_size=500,
    verbose=True             
)
synthesizer.fit(X)

c:\Users\dipen\OneDrive\Desktop\Qriocity\env\lib\site-packages\sdv\single_table\base.py:126: FutureWarning:

The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.

c:\Users\dipen\OneDrive\Desktop\Qriocity\env\lib\site-packages\sdv\single_table\base.py:105: UserWarning:

We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.



PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name            Est # of Columns (CTGAN)
Sequence_Length                 11
Basic_residues                  11
Acidic_residues                 11
Hydrophobic_residues            11
Extinction_Coefficient_cystines 11
Source_encoded                  11
Family_encoded                  11
ae                              11
aik                             11
akk                             11
akn                             11
aks                             11
an                              11
as                              11
c                               11
ce                              11
cf                              11
cs                              11
cw                              11
de                              11
df                              11
dfk                             11
di                         

Gen. (-0.00) | Discrim. (-1.53): 100%|██████████| 50/50 [02:05<00:00,  2.52s/it]


In [5]:
synthetic_data = synthesizer.sample(num_rows=2500)

augmented_data = pd.concat([data, synthetic_data], ignore_index=True)

print(f"Original Data Shape: {data.shape}")
print(f"Synthetic Data Shape: {synthetic_data.shape}")
print(f"Augmented Data Shape: {augmented_data.shape}")

Original Data Shape: (2342, 514)
Synthetic Data Shape: (2500, 262)
Augmented Data Shape: (4842, 514)


In [6]:
augmented_data.to_csv('augmented_dataset.csv', index=False)

print("Augmented data saved as 'augmented_dataset.csv'.")

Augmented data saved as 'augmented_dataset.csv'.
